# Playground

## Imports

In [1]:
import sys
from pathlib import Path
from fractions import Fraction

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from symbolica import Expression, S

from model import (
    COLOR_ADJ_INDEX,
    COLOR_FUND_INDEX,
    LORENTZ_INDEX,
    SPINOR_INDEX,
    WEAK_ADJ_INDEX,
    WEAK_FUND_INDEX,
    CovD,
    Field,
    FieldStrength,
    Gamma,
    GaugeFixing,
    GaugeGroup,
    GaugeRepresentation,
    GhostLagrangian,
    Model,
)
from model.lagrangian import (
    ComplexScalarKineticTerm,
    DiracKineticTerm,
    GaugeKineticTerm,
)
from symbolic.spenso_structures import (
    gauge_generator,
    structure_constant,
    weak_gauge_generator,
    weak_structure_constant,
)
from symbolic.vertex_engine import I


def expr_text(expr):
    text = expr.to_canonical_string() if hasattr(expr, "to_canonical_string") else str(expr)
    return text.replace("spenso::", "")


def signature_rows(signatures):
    return tuple(
        {
            "fields": ", ".join(signature.names),
            "arity": signature.arity,
            "terms": signature.term_count,
            "sectors": ", ".join(signature.sectors),
        }
        for signature in signatures
    )


def report_row(report):
    return {
        "matched_signatures": report.matched_signatures,
        "matched_terms": report.matched_terms,
        "total_signatures": report.total_signatures,
        "total_terms": report.total_terms,
        "signatures": signature_rows(report.signatures),
    }


def validation_rows(report):
    return {
        "ok": report.ok,
        "issues": tuple(
            {
                "code": issue.code,
                "severity": issue.severity,
                "message": issue.message,
            }
            for issue in report.issues
        ),
    }


## Symbols

In [2]:
mu, nu, rho = S("mu", "nu", "rho")
eQED, gS, g2, xiQCD = S("eQED", "gS", "g2", "xiQCD")
qPhi, qPsi, qQ = S("qPhi", "qPsi", "qQ")
lam, y, g4, gGamma = S("lam", "y", "g4", "gGamma")


## Fields

In [3]:
Phi = Field(
    "Phi",
    spin=0,
    self_conjugate=False,
    symbol=S("phi"),
    conjugate_symbol=S("phidag"),
    quantum_numbers={"Q": qPhi},
)
Chi = Field(
    "Chi",
    spin=0,
    self_conjugate=False,
    symbol=S("chi"),
    conjugate_symbol=S("chidag"),
)
Psi = Field(
    "Psi",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("psi"),
    conjugate_symbol=S("psibar"),
    indices=(SPINOR_INDEX,),
    quantum_numbers={"Q": qPsi},
)
Xi = Field(
    "Xi",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("xi"),
    conjugate_symbol=S("xibar"),
    indices=(SPINOR_INDEX,),
)
Quark = Field(
    "q",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("q"),
    conjugate_symbol=S("qbar"),
    indices=(SPINOR_INDEX, COLOR_FUND_INDEX),
    quantum_numbers={"Q": qQ},
)
Photon = Field("A", spin=1, self_conjugate=True, symbol=S("A0"), indices=(LORENTZ_INDEX,))
Gluon = Field("G", spin=1, self_conjugate=True, symbol=S("G0"), indices=(LORENTZ_INDEX, COLOR_ADJ_INDEX))
GhostG = Field(
    "ghG",
    spin=0,
    kind="ghost",
    self_conjugate=False,
    symbol=S("ghG0"),
    conjugate_symbol=S("ghGbar0"),
    indices=(COLOR_ADJ_INDEX,),
)
W = Field("W", spin=1, self_conjugate=True, symbol=S("W0"), indices=(LORENTZ_INDEX, WEAK_ADJ_INDEX))
GhostW = Field(
    "ghW",
    spin=0,
    kind="ghost",
    self_conjugate=False,
    symbol=S("ghW0"),
    conjugate_symbol=S("ghWbar0"),
    indices=(WEAK_ADJ_INDEX,),
)
H = Field(
    "H",
    spin=0,
    self_conjugate=False,
    symbol=S("H0"),
    conjugate_symbol=S("Hdag0"),
    indices=(WEAK_FUND_INDEX,),
)
L = Field(
    "L",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("L0"),
    conjugate_symbol=S("Lbar0"),
    indices=(SPINOR_INDEX, WEAK_FUND_INDEX),
)

(Phi, Chi, Psi, Xi, Quark, Photon, Gluon, GhostG, W, GhostW, H, L)


(Field(name='Phi', spin=0, self_conjugate=False, indices=(), kind='scalar', statistics='boson', symbol=phi, conjugate_symbol=phidag, mass=None, quantum_numbers={'Q': qPhi}, ghost_of=None, flavor_index=None, class_members=()),
 Field(name='Chi', spin=0, self_conjugate=False, indices=(), kind='scalar', statistics='boson', symbol=chi, conjugate_symbol=chidag, mass=None, quantum_numbers={}, ghost_of=None, flavor_index=None, class_members=()),
 Field(name='Psi', spin=Fraction(1, 2), self_conjugate=False, indices=(IndexType(name='Spinor', representation=Representation { rep: SelfDual(1), dim: Concrete(4) }, kind='spinor', dimension=None, is_flavor=False, prefix='i'),), kind='fermion', statistics='fermion', symbol=psi, conjugate_symbol=psibar, mass=None, quantum_numbers={'Q': qPsi}, ghost_of=None, flavor_index=None, class_members=()),
 Field(name='Xi', spin=Fraction(1, 2), self_conjugate=False, indices=(IndexType(name='Spinor', representation=Representation { rep: SelfDual(1), dim: Concrete(4

## Gauge Representations

In [4]:
COLOR_FUND_REP = GaugeRepresentation(
    index=COLOR_FUND_INDEX,
    generator_builder=gauge_generator,
    name="fund",
)
WEAK_DOUBLET_REP = GaugeRepresentation(
    index=WEAK_FUND_INDEX,
    generator_builder=weak_gauge_generator,
    name="doublet",
)

(COLOR_FUND_REP, WEAK_DOUBLET_REP)


(GaugeRepresentation(index=IndexType(name='ColorFund', representation=Representation { rep: Dualizable(3), dim: Concrete(3) }, kind='color_fund', dimension=None, is_flavor=False, prefix='c'), generator_builder=<function gauge_generator at 0x10ccf0930>, name='fund', slot=None, slot_policy='unique'),
 GaugeRepresentation(index=IndexType(name='WeakFund', representation=Representation { rep: Dualizable(3), dim: Concrete(2) }, kind='weak_fund', dimension=None, is_flavor=False, prefix='w'), generator_builder=<function weak_gauge_generator at 0x10ccf0a90>, name='doublet', slot=None, slot_policy='unique'))

## Gauge Groups

In [5]:
U1QED = GaugeGroup(
    name="U1QED",
    abelian=True,
    coupling=eQED,
    gauge_boson=Photon,
    charge="Q",
)
SU3C = GaugeGroup(
    name="SU3C",
    abelian=False,
    coupling=gS,
    gauge_boson=Gluon,
    ghost_field=GhostG,
    structure_constant=structure_constant,
    representations=(COLOR_FUND_REP,),
)
SU2L = GaugeGroup(
    name="SU2L",
    abelian=False,
    coupling=g2,
    gauge_boson=W,
    ghost_field=GhostW,
    structure_constant=weak_structure_constant,
    representations=(WEAK_DOUBLET_REP,),
)

(U1QED, SU3C, SU2L)


(GaugeGroup(name='U1QED', abelian=True, coupling=eQED, gauge_boson=Field(name='A', spin=1, self_conjugate=True, indices=(IndexType(name='Lorentz', representation=Representation { rep: InlineMetric(0), dim: Concrete(4) }, kind='lorentz', dimension=None, is_flavor=False, prefix='mu'),), kind='vector', statistics='boson', symbol=A0, conjugate_symbol=None, mass=None, quantum_numbers={}, ghost_of=None, flavor_index=None, class_members=()), ghost_field=None, structure_constant=None, representations=(), charge='Q'),
 GaugeGroup(name='SU3C', abelian=False, coupling=gS, gauge_boson=Field(name='G', spin=1, self_conjugate=True, indices=(IndexType(name='Lorentz', representation=Representation { rep: InlineMetric(0), dim: Concrete(4) }, kind='lorentz', dimension=None, is_flavor=False, prefix='mu'), IndexType(name='ColorAdj', representation=Representation { rep: SelfDual(2), dim: Concrete(8) }, kind='color_adj', dimension=None, is_flavor=False, prefix='a')), kind='vector', statistics='boson', symbol

## Local Models

In [6]:
local_yukawa_model = Model(y * Psi.bar * Psi * Phi)
local_four_fermion_model = Model(g4 * Psi.bar * Psi * Xi.bar * Xi)
local_quartic_model = Model(lam * Phi.bar * Phi * Chi.bar * Chi)
gamma_model = Model(
    (
        gGamma * Psi.bar * Gamma(mu) * Gamma(nu) * Psi
        + gGamma * Psi.bar * Gamma(nu) * Gamma(mu) * Psi
    ),
)

## Basic Model

In [7]:
basic_model = Model(
    (
        CovD(Phi.bar, mu) * CovD(Phi, mu)
        + I * Psi.bar * Gamma(mu) * CovD(Psi, mu)
        + I * Quark.bar * Gamma(mu) * CovD(Quark, mu)
        + (-(Expression.num(1) / Expression.num(4)) * FieldStrength(U1QED, mu, nu) * FieldStrength(U1QED, mu, nu))
        + (-(Expression.num(1) / Expression.num(4)) * FieldStrength(SU3C, mu, nu) * FieldStrength(SU3C, mu, nu))
        + GaugeFixing(SU3C, xi=xiQCD)
        + GhostLagrangian(SU3C)
    ),
    gauge_groups=(U1QED, SU3C),
)
compiled = basic_model.lagrangian()

{
    "term_count": len(compiled.terms),
    "signature_count": len(compiled.vertex_signatures()),
}


{'term_count': 18, 'signature_count': 14}

## Vertex Output

In [8]:
{
    "qbar_q_A": expr_text(compiled.feynman_rule(Quark.bar, Quark, Photon)),
    "qbar_q_G": expr_text(compiled.feynman_rule(Quark.bar, Quark, Gluon)),
    "psi_psi_A_no_delta": expr_text(compiled.feynman_rule(Psi.bar, Psi, Photon, include_delta=False)),
    "yukawa_unstripped": expr_text(local_yukawa_model.lagrangian().feynman_rule(Psi.bar, Psi, Phi, simplify=False, strip_externals=False)),
    "gamma_simplified": expr_text(gamma_model.lagrangian().feynman_rule(Psi.bar, Psi, simplify=True, simplify_gamma=True)),
}


{'qbar_q_A': '1𝑖*python::{}::eQED*python::{}::qQ*{symmetric,real}::g({upper}::cof(3,python::{}::c1),{upper}::cof(3,python::{}::c2))*{}::gamma({upper}::bis(4,python::{}::i1),{upper}::bis(4,python::{}::i2),{upper}::mink(4,python::{}::mu3))',
 'qbar_q_G': '1𝑖*python::{}::gS*{real}::t({upper}::coad(8,python::{}::a3),{upper}::cof(3,python::{}::c1),{upper}::cof(3,python::{}::c2))*{}::gamma({upper}::bis(4,python::{}::i1),{upper}::bis(4,python::{}::i2),{upper}::mink(4,python::{}::mu3))',
 'psi_psi_A_no_delta': '1𝑖*python::{}::eQED*python::{}::qPsi*{}::gamma({upper}::bis(4,python::{}::i1),{upper}::bis(4,python::{}::i2),{upper}::mink(4,python::{}::mu3))',
 'yukawa_unstripped': '1𝑖*exp(-1𝑖*python::{}::Dot(python::{}::q1+python::{}::q2+python::{}::q3,python::{}::x_))*python::{symmetric}::delta(python::{}::phi,python::{}::phi)*python::{symmetric}::delta(python::{}::psi,python::{}::psi)*python::{symmetric}::delta(python::{}::psibar,python::{}::psibar)*python::{}::U(python::{}::phi,python::{}::q3)*py

## Vertex Reports

In [9]:
{
    "arity_3": signature_rows(compiled.vertex_signatures(arity=3)),
    "exact_qbar_q_G": signature_rows(compiled.vertex_signatures(signature=(Quark.bar, Quark, Gluon))),
    "contains_ghosts": signature_rows(compiled.vertex_signatures(contains_fields=(GhostG.bar, GhostG))),
    "report_qbar_q_G": report_row(compiled.vertex_report(signature=(Quark.bar, Quark, Gluon))),
}


{'arity_3': ({'fields': 'G, G, G',
   'arity': 3,
   'terms': 1,
   'sectors': 'pure_gauge'},
  {'fields': 'Phi.bar, Phi, A', 'arity': 3, 'terms': 2, 'sectors': 'matter'},
  {'fields': 'Psi.bar, Psi, A', 'arity': 3, 'terms': 1, 'sectors': 'matter'},
  {'fields': 'ghG.bar, G, ghG', 'arity': 3, 'terms': 1, 'sectors': 'ghost'},
  {'fields': 'q.bar, q, A', 'arity': 3, 'terms': 1, 'sectors': 'matter'},
  {'fields': 'q.bar, q, G', 'arity': 3, 'terms': 1, 'sectors': 'matter'}),
 'exact_qbar_q_G': ({'fields': 'q.bar, q, G',
   'arity': 3,
   'terms': 1,
   'sectors': 'matter'},),
 'contains_ghosts': ({'fields': 'ghG.bar, ghG',
   'arity': 2,
   'terms': 1,
   'sectors': 'ghost'},
  {'fields': 'ghG.bar, G, ghG', 'arity': 3, 'terms': 1, 'sectors': 'ghost'}),
 'report_qbar_q_G': {'matched_signatures': 1,
  'matched_terms': 1,
  'total_signatures': 14,
  'total_terms': 18,
  'signatures': ({'fields': 'q.bar, q, G',
    'arity': 3,
    'terms': 1,
    'sectors': 'matter'},)}}

## Model Validation

In [10]:
bad_ghost_model = Model(
    GhostLagrangian(U1QED),
    gauge_groups=(U1QED,),
)
bad_rep_model = Model(
    ComplexScalarKineticTerm(field=Phi, gauge_group=SU3C),
    gauge_groups=(SU3C,),
)
bad_kinetic_model = Model(
    ComplexScalarKineticTerm(field=Phi, coefficient=2),
)

{
    "basic": validation_rows(basic_model.validate()),
    "bad_ghost": validation_rows(bad_ghost_model.validate()),
    "bad_rep": validation_rows(bad_rep_model.validate()),
    "bad_kinetic": validation_rows(bad_kinetic_model.validate()),
}


{'basic': {'ok': True, 'issues': ()},
 'bad_ghost': {'ok': False,
  'issues': ({'code': 'abelian_ghost_sector',
    'severity': 'error',
    'message': "Ghost validation only supports non-abelian gauge groups; got 'U1QED'."},)},
 'bad_rep': {'ok': False,
  'issues': ({'code': 'missing_gauge_representation',
    'severity': 'error',
    'message': "Covariant validation requires field 'Phi' to carry a declared representation under gauge group 'SU3C'. Field indices: (none). Declared representation indices: ColorFund."},)},
 'bad_kinetic': {'ok': False,
  'issues': ({'code': 'kinetic_normalization',
    'severity': 'error',
    'message': "complex-scalar kinetic term for field 'Phi' has non-canonical coefficient 2; expected 1."},)}}

## Compiled Validation

In [11]:
same_name_1 = Field("Phi", spin=0, self_conjugate=False, symbol=S("phi1"), conjugate_symbol=S("phi1dag"))
same_name_2 = Field("Phi", spin=0, self_conjugate=False, symbol=S("phi2"), conjugate_symbol=S("phi2dag"))
mass_mixing_model = Model(S("m12") * Phi.bar * Chi)
same_name_mixing_model = Model(S("mSame") * same_name_1.bar * same_name_2)
noncanonical_pair_model = Model(S("cNon") * Phi * Chi)

{
    "mass_mixing": validation_rows(mass_mixing_model.lagrangian().validate()),
    "same_name_mixing": validation_rows(same_name_mixing_model.lagrangian().validate()),
    "noncanonical_pair": validation_rows(noncanonical_pair_model.lagrangian().validate()),
}


{'mass_mixing': {'ok': True,
  'issues': ({'code': 'mass_structure_mixing',
    'severity': 'warning',
    'message': "Off-diagonal scalar mass-like bilinear detected between fields 'Phi' and 'Chi'; compiled term has 0 derivatives and only 2 matter fields."},)},
 'same_name_mixing': {'ok': True,
  'issues': ({'code': 'mass_structure_mixing',
    'severity': 'warning',
    'message': "Off-diagonal scalar mass-like bilinear detected between fields 'Phi' and 'Phi'; compiled term has 0 derivatives and only 2 matter fields."},)},
 'noncanonical_pair': {'ok': True, 'issues': ()}}

## Sector Counts

In [12]:
all_gg = [sig for sig in compiled.vertex_signatures() if sig.names == ("G", "G")]
pure_gg = [sig for sig in compiled.vertex_signatures(sector="pure_gauge") if sig.names == ("G", "G")]
gauge_fixing_gg = [sig for sig in compiled.vertex_signatures(sector="gauge_fixing") if sig.names == ("G", "G")]

{
    "GG_all": signature_rows(all_gg),
    "GG_pure_gauge": signature_rows(pure_gg),
    "GG_gauge_fixing": signature_rows(gauge_fixing_gg),
    "pure_gauge": report_row(compiled.vertex_report(sector="pure_gauge")),
    "gauge_fixing": report_row(compiled.vertex_report(sector="gauge_fixing")),
    "ghost": report_row(compiled.vertex_report(sector="ghost")),
}


{'GG_all': ({'fields': 'G, G',
   'arity': 2,
   'terms': 3,
   'sectors': 'gauge_fixing, pure_gauge'},),
 'GG_pure_gauge': ({'fields': 'G, G',
   'arity': 2,
   'terms': 2,
   'sectors': 'pure_gauge'},),
 'GG_gauge_fixing': ({'fields': 'G, G',
   'arity': 2,
   'terms': 1,
   'sectors': 'gauge_fixing'},),
 'pure_gauge': {'matched_signatures': 4,
  'matched_terms': 6,
  'total_signatures': 14,
  'total_terms': 18,
  'signatures': ({'fields': 'A, A',
    'arity': 2,
    'terms': 2,
    'sectors': 'pure_gauge'},
   {'fields': 'G, G', 'arity': 2, 'terms': 2, 'sectors': 'pure_gauge'},
   {'fields': 'G, G, G', 'arity': 3, 'terms': 1, 'sectors': 'pure_gauge'},
   {'fields': 'G, G, G, G', 'arity': 4, 'terms': 1, 'sectors': 'pure_gauge'})},
 'gauge_fixing': {'matched_signatures': 1,
  'matched_terms': 1,
  'total_signatures': 14,
  'total_terms': 18,
  'signatures': ({'fields': 'G, G',
    'arity': 2,
    'terms': 1,
    'sectors': 'gauge_fixing'},)},
 'ghost': {'matched_signatures': 2,
  'mat